In [ ]:
Content
1. Concurrency in a single system
    - When we see concurrency Issues
    - Issues and Fixes
    - Lock Code
2. Concurrency in a distributed system
3. SemaphoreSlim

# 1. Concurrency in a single system

In [ ]:
# When we see concurrency Issues:
    - On web server, each request runs on a seperate thread, so no issues there.
    - Concurrency Issues occurs only when Data is SHARED across requests/threads and modified.
- Example 1: Case 1: Shared In-Memory State
    - public static int counter = 0;
        public void Increment()
        {
            counter++;  // NOT thread-safe
        }
    - Issue: Multiple requests → same variable → race condition


- Example: Caching 
    - Thread A adds and thread B update or remove.
    - public class UserService
        {
            private static Dictionary<int, User> cache = new();
            public User GetUser(int id)
            {
                if (cache.ContainsKey(id))
                {
                    return cache[id]; // from cache
                }
        
                var user = GetFromDatabase(id);
                cache[id] = user; // store in cache
                return user;
            }
        }

- Example: Frontend
    - User clicks pay twice, 2 thread calls the same API

- Example: Distributed
    - Instance A, Instance B, Instance C: all access same redis/ DB

# How do you handle concurrency issues?
1. Identify shared mutable state
2. Check for race conditions
3. Choose approach:
   - Locks for critical sections
   - Atomic operations for simple updates
   - Immutability when possible
   - Concurrent collections for shared data
   - Queues to decouple systems
4. Ensure no deadlocks (lock ordering)
5. Consider performance vs safety trade-offs

In [ ]:
Multi-Threaded Env: Multiple threads are executing within the same process, often sharing memory, at the same time.

Share:
- Heap memory (objects, variables)
- Files, DB connections

Have separate:
- Stack (local variables, function calls)

# Concurrency Issues occurs only
1. Multiple threads
2. write operation on Shared resource + No coordination

Concurrency  -> managing multiple tasks (ccontext switching)
Parallelism  -> executing multiple tasks simultaneously (multi-core)


# 5 Types of Concurrency Issue:
1. Race Condition: Outcome depends on execution order
    - A race condition occurs when multiple threads access shared data and the final result depends on the timing/order of execution.
    - Even simple operations like counter++ are not atomic and can cause race conditions.
    - Example:
        counter = 0
        Thread A: read(0) → +1 → write(1)
        Thread B: read(0) → +1 → write(1)
    - Solution
        - Locks (mutex)
        - Atomic operations (Interlocked / CAS)
        

2. Deadlock: Both stuck forever
    - Deadlock happens when two or more threads are waiting on each other indefinitely.
    - Example:
        Thread A → holds Lock A → waiting for Lock B
        Thread B → holds Lock B → waiting for Lock A
    - Solution:
        - Lock ordering (always acquire locks in same order)
        - Use timeout (tryLock)
        - Avoid nested locks

3. Starvation: Low priority never gets resources.
    - A thread never gets CPU/resources because others keep taking them.
    - Example: High priority threads always run, and Low priority thread never executes
    - Solution:
        - Fair locks
        - Priority balancing
    - Solution:
        - Fair locks
        - Priority balancing

4. Livelock: Threads keep reacting to each other but no progress
    - Threads keep changing state in response to each other but never make progress.
    - Livelock → active but useless, and Deadlock → stuck
    - Example: Thread A: "You go first" Thread B: "No, you go first"
    - Solution:
        - Random backoff
        - Retry with delay

5. Visibility Issue: Thread A updates variable but Thread B doesn’t see updated value. May be Because of CPU cache.
    - One thread updates a variable, but another thread doesn’t see the updated value.
    - Root cause: CPU Memory caching
    - Solution: - locks (implicit visibility guarantee)

# Concurrency Problem is solved with:
1. Locks (mutex)
    - Only one thread enters critical section
    - Mutex (Mutual Exclusion) is a synchronization mechanism that ensures only one thread can access a critical section at a time.
    - C# example:
                lock (_lock)
                {
                    counter++;
                }
    - Cons: Performance overhead, can cause deadlocks


2. Atomic operations
    - Executes operation as a single unit (no interruption)
    - C example: Interlocked.Increment(ref counter);
    - Prefer atomic operations for simple counters instead of locks
    - Cons: limited usecase

3. Immutability
    - Objects never change → no synchronization needed
    - Immutable objects eliminate entire class of concurrency bugs
    - Cons: Memory overhead

4. Thread-safe data structures
    - Handle synchronization internally, these are prefered over locks
    - Example: ConcurrentDictionary (C#)
    - Cons: Less control

        
5. Queues (decoupling)
    - Instead of shared state → use message passing
    - Queues are preferred in distributed systems to avoid shared state issues
    - Cons: Complexity, Complexity


# Key Terms:
1. Critical Section: a piece of code, where we are trying to access share resource.
2. Process: one process can have multiple threads.
    

In [ ]:
# Solving with lock
LOCK:
    Read
    Check
    Update
UNLOCK

Example: Thread A enters lock → others blocked
         Thread A exits → next thread enters

# C# example
public class SeatService
{
    private readonly object _lock = new object();
    private bool isBooked = false;

    public void BookSeat()
    {
        lock (_lock) // critical section
        {
            if (!isBooked)
            {
                Console.WriteLine("Booking seat...");
                isBooked = true;
            }
            else
            {
                Console.WriteLine("Already booked");
            }
        }
    }
}


# Python example
import threading

lock = threading.Lock()
is_booked = False

def book_seat():
    global is_booked
    
    with lock:  # critical section
        if not is_booked:
            print("Booking seat...")
            is_booked = True
        else:
            print("Already booked")

# 2. Concurrency in a distributed system

In [ ]:
# Concurrency Control in Distributed System
When multiple servers running the same service.
Problem: Many concurrent request tries to book same Movie theatre Seat

# Important: This is majorly solved with the following, check DB sheets for more details.
    - Optimistic Concurrency Control (OCC)
    - Pessimistic Concurrency Control (PCC)

# Problems:
1. Lost Updates
    - No synchronization across services
    - Example:  Service A → reads stock = 1
                Service B → reads stock = 1
                Both update → stock = 0 (twice sold ❌)
    - Solutions:
        - DB transaction, Atomic operations

2. Duplicate Requests (Idempotency Problem)
    - Network retries are common in distributed systems
    - Solution: Idempotency key
3. Distributed Locking Problem
    - Multiple servers try to assign same driver, normal Locks does not works accross multiple machines.
    - Solutions: Redis distributed lock, ZooKeeper
4. Stale Data / Eventual Consistency
    - because of Replication delay
    - Example: User updates profile → Service A updated and → Service B still has old value
    - Solution: Use strong consistency where critical
5. Cache Inconsistency
    - DB updated but Cache not updated, so User sees stale data
    - Solution: Cache invalidation
6. Message Duplication
    - Message processed twice or processed in wrong order
    - solution: Idempotent consumers
7. Distributed Deadlocks
    - Service A waits from Service B, and Service B waits from Service A.
    - Solution: Timeout, Circuit Breakers


# 3. SemaphoreSlim

In [ ]:
# Semaphore
Semaphore is a variable (integer counter) used in concurrent programming to control/ limit access to shared resources.
- It has 2 atomic operations:
    wait()   → decrement (acquire)
    signal() → increment (release)


SemaphoreSlim controls concurrency by allowing only a fixed number of threads to access a resource simultaneously.
var semaphore = new SemaphoreSlim(2); // only 2 tasks at a time

Only 2 tasks run at same time:

Task 1, Task 2 → running
Task 3 waits
Task 4 waits    



Feature         Semaphore              Mutex
-------------------------------------------------------
Count           Multiple (N)           Only 1
Async           yes                    No
Ownership       No ownership           Has owner thread
Use case        Limited resources      Critical section
Release         Any thread             Only owner thread

# Example usecase of controlled parallelism
    API Gateway
     ↓
Rate Service
     ↓
Parallel Calls:
   → Airline A API
   → Airline B API
   → Airline C API
     ↓
Aggregate + return response


# C# Implementation
    public async Task<List<Rate>> GetRatesAsync(string origin, string destination)
{
    var airlines = new List<string> { "A", "B", "C", "D", "E" };

    var semaphore = new SemaphoreSlim(3); // max 3 concurrent calls

    var tasks = airlines.Select(async airline =>
    {
        await semaphore.WaitAsync();

        try
        {
            return await GetRateFromAirline(airline, origin, destination);
        }
        catch
        {
            return null; // handle failure
        }
        finally
        {
            semaphore.Release();
        }
    });

    var results = await Task.WhenAll(tasks);

    return results.Where(r => r != null).ToList();
}

# Pros:
    - If one airline fails, still return others
    - Seamaphore limit per airline
            { "AirlineA", new SemaphoreSlim(2) },
            { "AirlineB", new SemaphoreSlim(5) },

# key points:
    - Add circuit breaker, if airline is down.




# How to call Multiple Airline APIs in parallel
 - We may asynchronous calls with Task.WhenAll to call multiple airline APIs concurrently. To avoid overwhelming external systems or hitting rate limits, I would use SemaphoreSlim to limit the number of concurrent requests. 
    Each API call would have a timeout and retry mechanism, and I would aggregate the results, allowing partial failures. 
    Additionally, I would use caching to reduce repeated calls and improve performance.

# Without Seamphore
public async Task<List<Rate>> GetRatesAsync(string origin, string destination)
{
    var tasks = new List<Task<Rate>>
    {
        GetRateFromAirlineA(origin, destination),
        GetRateFromAirlineB(origin, destination),
        GetRateFromAirlineC(origin, destination)
    };

    var results = await Task.WhenAll(tasks);

    return results.Where(r => r != null).ToList();
}